## Lưu dữ liệu vào file txt

In [2]:
!pip install pdfplumber

  Using cached pdfplumber-0.11.7-py3-none-any.whl.metadata (42 kB)
  Using cached pdfminer_six-20250506-py3-none-any.whl.metadata (4.2 kB)
  Using cached pypdfium2-4.30.0-py3-none-win_amd64.whl.metadata (48 kB)
  Using cached cryptography-45.0.7-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached cffi-1.17.1-cp313-cp313-win_amd64.whl.metadata (1.6 kB)
  Using cached pycparser-2.22-py3-none-any.whl.metadata (943 bytes)
Using cached pdfplumber-0.11.7-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20250506-py3-none-any.whl (5.6 MB)
Using cached cryptography-45.0.7-cp311-abi3-win_amd64.whl (3.4 MB)
Using cached cffi-1.17.1-cp313-cp313-win_amd64.whl (182 kB)
Using cached pypdfium2-4.30.0-py3-none-win_amd64.whl (2.9 MB)
Using cached pycparser-2.22-py3-none-any.whl (117 kB)

   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   ---------------------------------------- 0/6 [pypdfium2]
   ------ ---------------------


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain.document_loaders import PyPDFLoader

pdf_path = r"E:\EduRAGBot\backend\data\raw\Quy-che-32-ngay-05-1-2017-Quy-che-Cong-tac-sinh-vien-tai-Dai-hoc-Quoc-gia-Ha-Noi (1).pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

full_text = "\n".join([doc.page_content for doc in documents])

output_path = "quy_che_cong_tac_sinh_vien.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print(f"file path: {output_path}")

file path: quy_che_cong_tac_sinh_vien.txt


## Chia chunk cho qcdt

In [2]:
import json

def chunk_document(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        if len(data) == 0:
            raise ValueError("không tìm thấy file")
        data = data[0]

    chunks = []

    # 1. Chunk cho document_info
    doc_info = data.get("document_info", {})
    if doc_info:
        info_text = []
        for k, v in doc_info.items():
            info_text.append(f"{k}: {v}")
        chunks.append({
            "article_number": None,
            "article_title": "Thông tin văn bản",
            "clause_number": None,
            "chunk_content": "\n".join(info_text)
        })

    # 2. Chunk cho preamble
    preamble = data.get("preamble", [])
    if preamble:
        chunks.append({
            "article_number": None,
            "article_title": "Căn cứ pháp lý",
            "clause_number": None,
            "chunk_content": "\n".join(preamble)
        })

    # 3. Chunks cho articles
    articles = data.get("articles", [])
    if not articles:
        print("Không tìm thấy articles")

    for article in articles:
        article_number = article.get("article_number")
        article_title = article.get("article_title")

        for clause in article.get("clauses", []):
            clause_number = clause.get("clause_number")
            content = clause.get("content", "")

            points = clause.get("points", [])
            if points:
                for p in points:
                    content += "\n\n" + p.get("content", "")

            chunks.append({
                "article_number": article_number,
                "article_title": article_title,
                "clause_number": clause_number,
                "chunk_content": content.strip()
            })

    # Xuất ra file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Đã chia {len(chunks)} chunks và lưu vào {output_file}")


# Ví dụ chạy
if __name__ == "__main__":
    chunk_document("Test.json", "chunks.json")


Đã chia 220 chunks và lưu vào chunks.json


In [ ]:
import json
from sentence_transformers import SentenceTransformer

def create_embeddings(input_file, output_file, model_name="all-MiniLM-L6-v2"):
    # Load model embedding
    print(f"Đang tải model embedding: {model_name}")
    model = SentenceTransformer(model_name)

    # Đọc dữ liệu chunk
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Tạo embedding cho từng chunk
    for chunk in chunks:
        text = chunk.get("chunk_content", "")
        embedding = model.encode(text).tolist()  # Chuyển numpy array -> list để lưu JSON
        chunk["embedding"] = embedding

    # Lưu kết quả ra file mới
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Đã tạo embedding cho {len(chunks)} chunks và lưu vào {output_file}")


# Ví dụ chạy
if __name__ == "__main__":
    create_embeddings("chunks_2.json", "chunks_with_embeddings_2.json")

## chia chunk cho qcctsv

In [ ]:
import json

def chunk_document(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    chunks = []

    # Xử lý phần decision
    decision = data.get("decision", {})
    if decision:
        # Thông tin cơ bản
        info_text = []
        for k, v in decision.items():
            if isinstance(v, (str, int)):
                info_text.append(f"{k}: {v}")
        if info_text:
            chunks.append({
                "article_number": None,
                "article_title": "Thông tin Quyết định",
                "clause_number": None,
                "chunk_content": "\n".join(info_text)
            })

        # Preamble
        preamble = decision.get("preamble", [])
        if preamble:
            chunks.append({
                "article_number": None,
                "article_title": "Căn cứ pháp lý (Quyết định)",
                "clause_number": None,
                "chunk_content": "\n".join(preamble)
            })

        # Articles trong decision
        articles = decision.get("articles", [])
        for article in articles:
            chunks.append({
                "article_number": article.get("article_number"),
                "article_title": "Điều trong Quyết định",
                "clause_number": None,
                "chunk_content": article.get("content", "")
            })

    # Xử lý phần regulation
    regulation = data.get("regulation", {})
    if regulation:
        # Thông tin cơ bản
        info_text = []
        for k, v in regulation.get("header", {}).items():
            info_text.append(f"{k}: {v}")
        if regulation.get("title"):
            info_text.append(f"Title: {regulation['title']}")
        if regulation.get("enactment_info"):
            info_text.append(f"Enactment: {regulation['enactment_info']}")
        if info_text:
            chunks.append({
                "article_number": None,
                "article_title": "Thông tin Quy chế",
                "clause_number": None,
                "chunk_content": "\n".join(info_text)
            })

        # Duyệt chapters → articles → clauses → points
        chapters = regulation.get("chapters", [])
        for chapter in chapters:
            chapter_title = chapter.get("chapter_title", "Chương")
            for article in chapter.get("articles", []):
                article_number = article.get("article_number")
                article_title = article.get("article_title", chapter_title)

                for clause in article.get("clauses", []):
                    clause_number = clause.get("clause_number")
                    content = clause.get("content", "")

                    # Nếu có points → nối vào sau content
                    points = clause.get("points", [])
                    if points:
                        point_texts = []
                        for p in points:
                            letter = p.get("point_letter", "")
                            p_content = p.get("content", "")
                            if letter:
                                point_texts.append(f"{letter}) {p_content}")
                            else:
                                point_texts.append(p_content)
                        content += "\n" + "\n".join(point_texts)

                    chunks.append({
                        "article_number": article_number,
                        "article_title": article_title,
                        "clause_number": clause_number,
                        "chunk_content": content.strip()
                    })

    # Xử lý appendix
    appendix = data.get("appendix", [])
    if appendix:
        for clause in appendix:
            chunks.append({
                "article_number": None,
                "article_title": "Phụ lục",
                "clause_number": clause.get("clause_number"),
                "chunk_content": clause.get("content", "").strip()
            })

    # Xuất ra file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"chia {len(chunks)} chunks vào {output_file}")


if __name__ == "__main__":
    chunk_document("quy_che_cong_tac_sinh_vien.json", "chunks_2.json")


chia 155 chunks vào chunks_2.json


In [6]:
import json
from sentence_transformers import SentenceTransformer

def create_embeddings(input_file, output_file, model_path="E:\\EduRAGBot\\backend\\data\\dpr-phobert-augmented"):
    print(f"Đang tải model embedding từ: {model_path}")
    model = SentenceTransformer(model_path)

    # Đọc dữ liệu chunk
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Tạo embedding cho từng chunk
    for chunk in chunks:
        text = chunk.get("chunk_content", "").strip()
        if text:  # chỉ encode nếu không rỗng
            embedding = model.encode(text).tolist()  # chuyển numpy sang list để lưu json
            chunk["embedding"] = embedding
        else:
            chunk["embedding"] = []

    # Lưu kết quả ra file mới
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"tạo embedding cho {len(chunks)} chunks vào {output_file}")


if __name__ == "__main__":
    create_embeddings(
        "chunks_2.json",
        "chunks_with_embeddings_2.json",
        model_path="E:\\EduRAGBot\\backend\\data\\dpr-phobert-augmented"
    )


e:\EduRAGBot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang tải model embedding từ: E:\EduRAGBot\backend\data\dpr-phobert-augmented
tạo embedding cho 155 chunks vào chunks_with_embeddings_2.json


In [ ]:
import json
import numpy as np
import faiss
import pickle

def build_faiss_index(input_file, faiss_index_file="index_2.faiss", metadata_file="index_2.pkl"):
    # Đọc file chunks có embeddings
    with open(input_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    # Lấy danh sách embeddings và metadata
    embeddings = [chunk["embedding"] for chunk in chunks]
    metadata = [
        {
            "article_number": chunk.get("article_number"),
            "article_title": chunk.get("article_title"),
            "clause_number": chunk.get("clause_number"),
            "chunk_content": chunk.get("chunk_content"),
        }
        for chunk in chunks
    ]

    embeddings = np.array(embeddings).astype("float32")

    # Khởi tạo FAISS index dùng cosine similarity
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    faiss.normalize_L2(embeddings)

    # Thêm embeddings vào index
    index.add(embeddings)

    faiss.write_index(index, faiss_index_file)

    with open(metadata_file, "wb") as f:
        pickle.dump(metadata, f)

    print(f"Đã build FAISS index với {len(metadata)} vectors")
    print(f"   FAISS: {faiss_index_file}")
    print(f"   Metadata pickle: {metadata_file}")

if __name__ == "__main__":
    build_faiss_index(
        "chunks_with_embeddings_2.json",
        )


Đã build FAISS index với 155 vectors
   FAISS: index_2.faiss
   Metadata pickle: index_2.pkl


In [9]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ===== Load index + metadata =====
def load_faiss_index(faiss_index_file="index_2.faiss", metadata_file="index_2.pkl"):
    # Load FAISS index
    index = faiss.read_index(faiss_index_file)

    # Load metadata
    with open(metadata_file, "rb") as f:
        metadata = pickle.load(f)

    return index, metadata


# ===== Search trong FAISS =====
def search_faiss(query, index, metadata, model, top_k=5):
    # Encode query thành vector
    query_vec = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_vec)

    # Tìm top_k vectors gần nhất
    distances, indices = index.search(query_vec, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx == -1:  # Nếu không tìm thấy
            continue
        results.append({
            "rank": i+1,
            "score": float(distances[0][i]),
            "metadata": metadata[idx]
        })

    return results


# ===== Ví dụ chạy =====
if __name__ == "__main__":
    # Load lại index + metadata
    index, metadata = load_faiss_index("index_2.faiss", "index_2.pkl")
    model_path="E:\\EduRAGBot\\backend\\data\\dpr-phobert-augmented"
    # Dùng đúng model đã build index
    model = SentenceTransformer(model_path)

    # Query thử
    query = "nhiệm vụ của hội đồng thi đua"
    results = search_faiss(query, index, metadata, model, top_k=3)

    # In kết quả
    for r in results:
        print(f"Rank {r['rank']} | Score: {r['score']:.4f}")
        print(f"Article {r['metadata'].get('article_number')} - {r['metadata'].get('article_title')}")
        print(f"Content: {r['metadata']['chunk_content'][:200]}...\n")


Rank 1 | Score: 0.5897
Article 34 - Hội đồng thi đua, khen thưởng và kỷ luật sinh viên
Content: Cơ cấu Hội đồng
a) Đối với các đơn vị đào tạo thành viên
Thủ trưởng đơn vị ra quyết định thành lập Hội đồng thi đua, khen thưởng và kỷ luật sinh viên. Nhiệm kỳ của Hội đồng theo năm học. Thành phần củ...

Rank 2 | Score: 0.5597
Article 34 - Hội đồng thi đua, khen thưởng và kỷ luật sinh viên
Content: Nhiệm vụ của Hội đồng
Hội đồng thi đua, khen thưởng và kỷ luật sinh viên tư vấn giúp thủ trưởng đơn vị triển khai công tác khen thưởng, kỷ luật đối với sinh viên và chịu sự chỉ đạo trực tiếp của thủ t...

Rank 3 | Score: 0.5170
Article 26 - Các bước đánh giá
Content: Đối với các đơn vị đào tạo trực thuộc
a) Hội đồng cấp bộ môn: Chủ tịch Hội đồng (Chủ nhiệm bộ môn hoặc Phó Chủ nhiệm bộ môn được Chủ nhiệm bộ môn ủy quyền), các ủy viên (trợ lý công tác sinh viên, trợ...

